In [36]:
"""
Reproduces Fig. 5 through Fig. 10 from Ratnakar, R.R. (2026), 
"Thermodynamics and equilibrium thermochemistry of ortho- and 
para-hydrogen: Integrating quantum mechanics with classical
EOS modeling", Int. J. Hydrogen Energy 245, 155702.
https://doi.org/10.1016/j.ijhydene.2026.155702.

Per the paper's own scope (Sec. 3.3: "For brevity of the demonstration
of the EOS model, only pH2 is considered in this section"), all Figure
reproductions here use para-hydrogen only.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator, LogLocator, 
    FixedLocator, FuncFormatter, FixedFormatter)
from h2_thermo_final import (R, EOS_PARAMS, eos_ac_b, a_of_T, molar_volume_PR,
    real_properties, sonic_velocity, joule_thomson)

OUTDIR = "figures"

Mw_H2 = 2.01588e-3  # [kg/mol]
SPECIES = "p"  # paper only demonstrates the residual/real-state part for pH2

# =======================================================================================================
# Vapor Liquid Equilibrium (VLE): fugacity coefficient (standard PR-78 result) and saturation P(T)
# =======================================================================================================
def fugacity_coeff(T, P, species, phase):
    """
    Fugacity coefficient phi. Phase equilibrium requires phi_liquid = phi_vapor.
    Fugacity is tied specifically to the residual Gibbs energy:
        G_res(T,P) = RT * ln(phi)
    The natural variables of G are T and P, but the PR-EOS is pressure-explicit,
    P = P(V,T). So we go through the residual Helmholtz energy A_res, whose
    natural variables (T,V) match the EOS, via the exact identity:
        ln(phi) = A_res/(R*T) + (Z - 1) - ln(Z), with:
        A_res(T,V) = −∫[P − RT/V] dV (integral goes from ∞ to V)
    To remember, PR-EOS:
        P = RT/(V−b) − a(T)/[(V+alpha1*b)(V+alpha1*b)]
    ------------------------------------------------------------------------------------
    Deriving A_res:
        A_res = R*T*ln[V/(V-b)] + a/((alpha1-alpha2)*b) * ln[(V+alpha2*b)/(V+alpha1*b)]
    Substituting V=Z*R*T/P and plugging into the ln(phi) identity above:
        ln(phi) = Z − 1 − ln(Z−B) − [A/(2√2·B)]·ln[(Z+(1+√2)B)/(Z+(1−√2)B)]
    """
    ac, b, m, Tc, Cpen = eos_ac_b(species)
    a = a_of_T(T, species)
    V_corr, Z = molar_volume_PR(T, P, species, phase)
    A = a * P / (R*T)**2
    B = b * P / (R*T)
    sqrt2 = np.sqrt(2)
    ln_phi = ((Z - 1) - np.log(Z - B)
              - (A/(2*sqrt2*B)) * np.log((Z + (1+sqrt2)*B) / (Z + (1-sqrt2)*B)))
    return np.exp(ln_phi)


def saturation_pressure(T, species, P_guess=None, tol=1e-10, max_iter=300):
    """Successive substitution: P_(n+1) = P_n * phi_liq/phi_vap, until phi_liq=phi_vap."""
    Pc = EOS_PARAMS[species]["Pc"]
    Tc = EOS_PARAMS[species]["Tc"]
    omega = EOS_PARAMS[species]["omega"]
    if P_guess is None:
        Tr = T / Tc
        P_guess = Pc * 10**(7.0/3.0*(1+omega)*(1-1/Tr))  # initial guess (Wilson's vapor pressure equation)
    P = P_guess
    for _ in range(max_iter):
        phi_l = fugacity_coeff(T, P, species, "liquid")
        phi_v = fugacity_coeff(T, P, species, "vapor")
        ratio = phi_l / phi_v
        P_new = P * ratio
        if abs(ratio - 1) < tol:
            return P_new
        P = P_new
    return P


# =======================================================================================================
# FIGURE 5a: P-T phase diagram (saturation curve from PR-78 VLE)
# =======================================================================================================
Tc = EOS_PARAMS[SPECIES]["Tc"]
Pc = EOS_PARAMS[SPECIES]["Pc"]
Tt = EOS_PARAMS[SPECIES]["Tt"]
Pt = EOS_PARAMS[SPECIES]["Pt"]

T_sat_grid = np.linspace(Tt, Tc - 0.02, 300)
P_sat_grid = []
P_prev = None
for T in T_sat_grid:
    P_prev = saturation_pressure(T, SPECIES, P_guess=P_prev)
    P_sat_grid.append(P_prev)
P_sat_grid = np.array(P_sat_grid)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot(T_sat_grid, P_sat_grid/1e5, color="#0066FF", label="Vapor-liquid (PR-78 EOS)")
ax.scatter([Tc], [Pc/1e5], color="#FF2D2D", zorder=5, label=f"Critical point ({Tc:.2f}, {Pc/1e5:.2f})")
ax.scatter([Tt], [Pt/1e5], color="#00C853", zorder=5, label=f"Triple point ({Tt:.2f}, {Pt/1e5:.4f})")
ax.set_yscale("log")
ax.set_xlim(5, 40)
ax.set_ylim(0.005, 150)
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))
ax.yaxis.set_minor_locator(LogLocator(subs="all"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{y:g}"))
ax.tick_params(axis="both", which="major", direction="in", length=3.6)
ax.tick_params(axis="both", which="minor", direction="in", length=2.2)
ax.set_xlabel("T, K")
ax.set_ylabel("P, bar")
ax.set_title("pH2 P-T phase diagram (reproduces Fig. 5a of the paper)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f"{OUTDIR}/fig5a_phase_diagram_PT.png", dpi=150)
plt.close(fig)
print("Saved fig5a_phase_diagram_PT.png")


# =======================================================================================================
# FIGURE 5b: P-V diagram + isotherms
# =======================================================================================================
def P_of_V_eos(V_eos, T, species):
    ac, b, m, Tc_, Cpen = eos_ac_b(species)
    a = a_of_T(T, species)
    alpha1, alpha2 = 1+np.sqrt(2), 1-np.sqrt(2)
    return R*T/(V_eos-b) - a/((V_eos+alpha1*b)*(V_eos+alpha2*b))

ac, b, m, _, Cpen = eos_ac_b(SPECIES)

# critical volume Vc
Vc_raw, _ = molar_volume_PR(Tc, Pc, SPECIES, phase="vapor")
Vc_cm3 = Vc_raw * 1e6  # [cm³/mol]

u = np.linspace(0, 1, 400)
T_sat_smooth = Tt + (Tc - 0.0001 - Tt) * (1 - (1 - u)**2)

V_liq_env, V_vap_env, P_sat_env = [], [], []
for Tsat in T_sat_smooth:
    try:
        Psat = saturation_pressure(Tsat, SPECIES)
        Vl, _ = molar_volume_PR(Tsat, Psat, SPECIES, phase="liquid")
        Vv, _ = molar_volume_PR(Tsat, Psat, SPECIES, phase="vapor")
        
        if Vv > Vl:
            V_liq_env.append(Vl * 1e6)
            V_vap_env.append(Vv * 1e6)
            P_sat_env.append(Psat / 1e5)
    except Exception:
        continue
V_envelope = np.concatenate([V_liq_env, [Vc_cm3], V_vap_env[::-1]])
P_envelope = np.concatenate([P_sat_env, [Pc / 1e5], P_sat_env[::-1]])
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.plot(V_envelope, P_envelope, color="#0066FF", lw=1.2, label="Vapor-Liquid Envelope (PR-78)")
ax.scatter([Vc_cm3], [Pc / 1e5], color="red", zorder=5, s=25, 
           label=f"Critical Point ({Vc_cm3:.1f} cm³/mol)")

# isotherms
Tc = EOS_PARAMS[SPECIES]["Tc"]
T_iso_list = [20, 25, 30, 32, Tc, 35, 50, 100, 200, 300]

for T_iso in T_iso_list:
    if T_iso < Tc:        
        Psat = saturation_pressure(T_iso, SPECIES)
        Vl, _ = molar_volume_PR(T_iso, Psat, SPECIES, phase="liquid")
        Vv, _ = molar_volume_PR(T_iso, Psat, SPECIES, phase="vapor")
    
        # liquid branch
        V_start = b + Cpen + 1e-9
        V_liq_eos_end = Vl + Cpen
        if V_liq_eos_end > V_start:
            V_liq_eos = np.geomspace(V_start, V_liq_eos_end, 500)
            P_liq = P_of_V_eos(V_liq_eos, T_iso, SPECIES)
            V_liq = (V_liq_eos - Cpen) * 1e6
            mask_liq = (P_liq > 0) & (P_liq < 200e5) & (V_liq > 0)
            ax.plot(V_liq[mask_liq], P_liq[mask_liq]/1e5, lw=0.6, color="gray", alpha=0.7)
    
        # two-phase region (constant pressure)
        Vl_phys = Vl * 1e6
        Vv_phys = Vv * 1e6
        if Vv_phys > Vl_phys:
            V_two_phase = np.linspace(Vl_phys, Vv_phys, 300)
            P_two_phase = np.full(V_two_phase.shape, Psat/1e5)
            ax.plot(V_two_phase, P_two_phase, lw=0.6, color="gray", alpha=0.7)
            
        # vapor branch
        V_max = 5e-2
        V_vap_eos_start = Vv + Cpen
        if V_vap_eos_start < V_max:
            V_vap_eos = np.geomspace(V_vap_eos_start, V_max, 1000)
            P_vap = P_of_V_eos(V_vap_eos, T_iso, SPECIES)
            V_vap = (V_vap_eos - Cpen) * 1e6
            mask_vap = (P_vap > 0) & (P_vap < 200e5) & (V_vap > 0)
            ax.plot(V_vap[mask_vap], P_vap[mask_vap]/1e5, lw=0.6, color="gray", alpha=0.7)

    else:
        V_min = b + Cpen + 1e-9
        V_max = 5e-2
        V_grid_eos = np.geomspace(V_min, V_max, 1500)
        P_iso = P_of_V_eos(V_grid_eos, T_iso, SPECIES)
        V_grid_corr = (V_grid_eos - Cpen) * 1e6
        mask = (P_iso > 0) & (P_iso < 200e5) & (V_grid_corr > 0)
        ax.plot(V_grid_corr[mask], P_iso[mask]/1e5, lw=0.6, color="gray", alpha=0.7)
        continue

ax.set_xscale('log')
ax.set_xlim(14, 1e4)
ax.xaxis.set_major_locator(FixedLocator([50, 100, 500, 1000, 5000, 1e4]))
ax.xaxis.set_major_formatter(FixedFormatter(['50', '100', '500', '1000', '5000', r"$10^4$"]))
ax.xaxis.set_minor_locator(LogLocator(base=10, subs=[3, 4, 6, 7, 8, 9]))
ax.set_ylim(0, 22)
ax.yaxis.set_major_locator(MultipleLocator(5))
ax.yaxis.set_minor_locator(MultipleLocator(1))
ax.tick_params(axis="both", which="major", direction="in", length=3.6)
ax.tick_params(axis="both", which="minor", direction="in", length=2.2)
ax.set_xlabel(r"V, cm$^3$/mol")
ax.set_ylabel("P, bar")
ax.set_title("pH2 P-V phase diagram(reproduces Fig. 5b of the paper)\n"
              "(gray lines = isotherms at 20, 25, 30, 32, Tc, 35, 50, 100, 200, 300K)", 
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(f"{OUTDIR}/fig5b_phase_diagram_PV.png", dpi=150)
plt.close(fig)
print("Saved fig5b_phase_diagram_PV.png")


# =======================================================================================================
# Now we have P and need to find Tsat(P)
# =======================================================================================================
def T_saturation(P_target, species, T_bracket=None):
    Tc_ = EOS_PARAMS[species]["Tc"]
    Tt_ = EOS_PARAMS[species]["Tt"]
    if P_target >= EOS_PARAMS[species]["Pc"]:
        return None  # supercritical pressure, no VLE
    f = lambda T: saturation_pressure(T, species) - P_target
    lo, hi = Tt_, Tc_ - 0.01
    try:
        return brentq(f, lo, hi, xtol=1e-6)
    except ValueError:
        return None


# =======================================================================================================
# FIGURES 6-10: density, Cp, Cv, sonic velocity, Joule-Thomson vs T,
# at P = 0.1, 1, 10, 20 bar (pH2 only, matching paper's Figs.6-10)
# =======================================================================================================
pressures_bar = [0.1, 1, 10, 20]

def build_T_grid(Tsat):
    coarse = np.geomspace(15, 600, 200)
    if Tsat is None or not (15 < Tsat < 600):
        return np.sort(coarse)
    # dense clusters approaching Tsat from below and above, log-spaced so
    # most points concentrate very close to Tsat (down to 1e-5 K away)
    below = Tsat - np.geomspace(0.1, min(5.0, Tsat - 15), 80)
    above = Tsat + np.geomspace(0.1, min(5.0, 600 - Tsat), 80)
    grid = np.concatenate([coarse, below, above])
    grid = grid[(grid > 15) & (grid < 600)]
    return np.unique(np.sort(grid))

def scan_property(P_bar, prop_func):
    P = P_bar * 1e5
    Tsat = T_saturation(P, SPECIES)
    T_out, V_out = [], []
    T_grid = build_T_grid(Tsat)
    for T in T_grid:
        if Tsat is not None and T < Tsat:
            phase = "liquid"
        else:
            phase = "vapor"
        try:
            val = prop_func(T, P, phase)
        except Exception:
            val = np.nan
        T_out.append(T)
        V_out.append(val)
    return np.array(T_out), np.array(V_out), Tsat


def get_density(T, P, phase):
    props = real_properties(T, P, SPECIES, phase=phase)
    return Mw_H2 / props["V_corr"]

def get_Cp_over_R(T, P, phase):
    return real_properties(T, P, SPECIES, phase=phase)["Cp"] / R

def get_Cv_over_R(T, P, phase):
    return real_properties(T, P, SPECIES, phase=phase)["Cv"] / R

def get_sonic(T, P, phase):
    return sonic_velocity(T, P, SPECIES, phase=phase)

def get_JT(T, P, phase):
    return joule_thomson(T, P, SPECIES, phase=phase) * 1e5  # [K/Pa] -> [K/bar]


def make_4panel_figure(prop_func, ylabel, title, fname, y_specs):
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, P_bar, tag, (ymin, ymax, ystep) in zip(axes.flat, pressures_bar,
                                                   ["(a)", "(b)", "(c)", "(d)"], y_specs):
        T_out, V_out, Tsat = scan_property(P_bar, prop_func)
        ax.plot(T_out, V_out, color="tab:orange", lw=1.5)
        if Tsat is not None:
            ax.axvline(Tsat, color="gray", ls=":", lw=1)
            
        # X axis
        ax.set_xscale('log')
        ax.set_xlim(14, 700)
        ax.xaxis.set_major_locator(FixedLocator([20, 50, 100, 200, 500]))
        ax.xaxis.set_major_formatter(FixedFormatter(['20', '50', '100', '200', '500']))
        ax.xaxis.set_minor_locator(LogLocator(base=10, subs=[3, 4, 6, 7, 8, 9]))
        # Y axis
        ax.set_ylim(ymin, ymax)
        ax.yaxis.set_major_locator(MultipleLocator(ystep))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))
        ax.tick_params(axis="both", which="major", direction="in", length=3.6)
        ax.tick_params(axis="both", which="minor", direction="in", length=2.2)
        
        ax.set_xlabel("T, K")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{tag} P = {P_bar} bar", loc="left", fontweight="bold")
        ax.grid(alpha=0.3)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(f"{OUTDIR}/{fname}", dpi=150)
    plt.close(fig)
    print(f"Saved {fname}")


make_4panel_figure(
    get_density, r"$\rho$, kg/m$^3$",
    "pH2 density vs T (reproduces Fig. 6 of the paper)",
    "fig6_density.png",
    y_specs=[(0, 0.20, 0.05), (0, 80, 20), (0, 80, 20), (0, 80, 20)],
)

make_4panel_figure(
    get_Cp_over_R, r"$C_{\mathrm{P}}/R$",
    "pH2 isobaric heat capacity vs T (reproduces Fig. 7 of the paper)",
    "fig7_Cp_real.png",
    y_specs=[(2.2, 4.2, 0.5), (2.2, 4.2, 0.5), (0, 13, 2), (0, 14, 2)],
)

make_4panel_figure(
    get_Cv_over_R, r"$C_{\mathrm{V}}/R$",
    "pH2 isochoric heat capacity vs T (reproduces Fig. 8 of the paper)",
    "fig8_Cv_real.png",
    y_specs=[(1.2, 3.2, 0.5)] * 4,
)

make_4panel_figure(
    get_sonic, "sonic velocity, m/s",
    "pH2 sonic velocity vs T (reproduces Fig. 9 of the paper)",
    "fig9_sonic_velocity.png",
    y_specs=[(100, 1900, 500), (100, 2100, 500), (100, 2200, 500), (100, 2300, 500)],
)

make_4panel_figure(
    get_JT, r"$\mu_{\mathrm{JT}}$, K/bar",
    "pH2 Joule-Thomson coefficient vs T (reproduces Fig. 10 of the paper)",
    "fig10_joule_thomson.png",
    y_specs=[(-0.2, 2.5, 0.5), (-0.3, 1.6, 0.5), (-0.2, 1.1, 0.2), (-0.12, 0.52, 0.1)],
)

print("\nAll figures done.")

Saved fig5a_phase_diagram_PT.png
Saved fig5b_phase_diagram_PV.png
Saved fig6_density.png
Saved fig7_Cp_real.png
Saved fig8_Cv_real.png
Saved fig9_sonic_velocity.png
Saved fig10_joule_thomson.png

All figures done.
